In [ ]:
import os
from glob import glob

import pandas

# Fathom flooding
- parse list of files
- output merge commands
- metadata from merged files

In [ ]:
df = pandas.read_csv('fathom_data/details.csv', names=['path'])
df

In [ ]:
def parse_type(s):
    path = s.path
    fname = os.path.basename(path)
    hazard, rp = fname.replace(".tif", "").split("_1in")
    return hazard, int(rp)
df[['hazard', 'rp']] = df.apply(parse_type, axis=1, result_type='expand')

In [ ]:
def parse_country(path):
    country = os.path.basename(os.path.dirname(os.path.dirname(path)))
    return country.replace("-", " ").title().replace("Of", "of")
df['country'] = df.path.apply(parse_country)

In [ ]:
rps = sorted(df.rp.unique())
hazards = sorted(df.hazard.unique())
rps, hazards

In [ ]:
with open('fathom_data/merge.sh', 'w') as fh:
    for rp in rps:
        for hazard in hazards:
            if hazard == 'FD':
                continue
            paths = list(df[(df.rp == rp) & (df.hazard == hazard)].path)
            cmd = f"""gdal_merge.py -co "COMPRESS=LZW" -o {hazard}__rp_{rp}.tif {' '.join(paths)}\n"""
            fh.write(cmd)

In [ ]:
merged_fnames = sorted(glob('fathom_data/*.tif'))

In [ ]:
floods = pandas.DataFrame({'path': merged_fnames})
def parse_merged(s):
    path = s.path
    # FU__rp_10
    fname = os.path.basename(path)
    hazard, rp = fname.replace(".tif", "").split("__rp_")
    return hazard, int(rp)
floods[['hazard', 'rp']] = floods.apply(parse_merged, axis=1, result_type='expand')
floods['ssp'] = 'historical'
floods.to_csv('details_fathom.csv', index=False)
floods

# Extreme heat
- parse meta from nc files

In [ ]:
heat_fnames = sorted(glob('world*/*.nc'))

In [ ]:
heat = pandas.DataFrame({'path':heat_fnames})
heat[['hazard','ssp','quantile','epoch']] = \
    heat.path.str.extract(
        r'climatology-(.*)-annual-mean_cmip6_annual_all-regridded-bct-(.*)-climatology_(.*)_(.*).nc', 
        expand=True)

heat.to_csv('details_heat.csv', index=False)
heat